# MuZero — Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model (2019)

_Papers · Reinforcement Learning_

**Learn three small networks — a state encoder, a latent dynamics model, and a policy/value head — so Monte-Carlo Tree Search can plan inside an imagined model, without ever being told the environment's rules.**

---

This notebook is a practice scaffold for the **MuZero — Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, math
torch.manual_seed(0); np.random.seed(0)

# ===== TOY TASK: corridor cells 0..6, START at 3. a=0 LEFT, a=1 RIGHT.
# GOAL = cell 6 -> reward +1, episode ends. Cell 0 = DEAD end -> reward 0, ends.
# The agent is NEVER told this rule. Self-play moves 50/50, so the POLICY PRIOR is
# uninformative -- only PLANNING through the learned dynamics can solve it.
N, START, GOAL, DEAD = 7, 3, 6, 0
GAMMA, D, A, K = 0.95, 8, 2, 4   # latent dim D, actions A, unroll depth K
def true_step(p, a):
    p2 = min(N-1, max(0, p + (1 if a == 1 else -1)))
    r  = 1.0 if p2 == GOAL else 0.0
    return p2, r, (p2 == GOAL or p2 == DEAD)
def oh(i, n):
    v = torch.zeros(n); v[i] = 1.; return v

# ===== The three learned functions of MuZero =====
class MuZero(nn.Module):
    def __init__(self):
        super().__init__()
        self.h   = nn.Sequential(nn.Linear(N, 32), nn.ReLU(), nn.Linear(32, D), nn.Tanh())  # representation
        self.g   = nn.Sequential(nn.Linear(D+A, 32), nn.ReLU(), nn.Linear(32, D), nn.Tanh())# dynamics -> next state
        self.rew = nn.Sequential(nn.Linear(D+A, 32), nn.ReLU(), nn.Linear(32, 1))           # dynamics -> reward
        self.pol = nn.Linear(D, A); self.val = nn.Linear(D, 1)                              # prediction f
    def repr(self, o):     return self.h(o)                                  # s0 = h(obs)
    def dyn(self, s, a):                                                     # r^k, s^k = g(s^{k-1}, a^k)
        x = torch.cat([s, a], -1); return self.g(x), self.rew(x).squeeze(-1)
    def pred(self, s):     return self.pol(s), self.val(s).squeeze(-1)       # p^k, v^k = f(s^k)

mz = MuZero(); opt = torch.optim.Adam(mz.parameters(), lr=4e-3)

def make_episode():                      # unbiased 50/50 self-play -> prior carries no direction
    p, ep = START, []
    for _ in range(20):
        a = 1 if np.random.rand() < 0.5 else 0
        p2, r, d = true_step(p, a); ep.append((p, a, r)); p = p2
        if d: break
    return ep
def nstep(ep, i, n=8):                    # value target z = n-step return
    G, disc = 0., 1.
    for j in range(i, min(i+n, len(ep))): G += disc*ep[j][2]; disc *= GAMMA
    return G

# ===== Train: unroll K steps, match reward+value+policy (Eq. 1) =====
def train(unroll=True, iters=4000):
    torch.manual_seed(0); np.random.seed(0)
    m = MuZero(); op = torch.optim.Adam(m.parameters(), lr=4e-3)
    for _ in range(iters):
        ep = make_episode()
        if not ep: continue
        s = m.repr(oh(ep[0][0], N).unsqueeze(0)); L = 0.
        steps = min(K, len(ep)) if unroll else 1     # ablation: steps=1 -> g never rolled forward
        for k in range(steps):
            logits, v = m.pred(s)
            L = L + F.mse_loss(v, torch.tensor([nstep(ep, k)]))                 # value loss l^v
            a = ep[k][1]
            pt = torch.tensor([[0.1, 0.9] if a == 1 else [0.9, 0.1]])          # policy target (visit-count proxy)
            L = L + 0.3*F.kl_div(F.log_softmax(logits, -1), pt, reduction='batchmean')  # policy loss l^p
            s, r = m.dyn(s, oh(a, A).unsqueeze(0))
            L = L + F.mse_loss(r, torch.tensor([ep[k][2]]))                    # reward loss l^r
        op.zero_grad(); L.backward(); op.step()
    return m
mz = train(unroll=True)

# ===== MCTS in the LEARNED latent model (pUCT, expand via g+f, value backup) =====
class Node:
    def __init__(self, prior): self.prior=prior; self.N=0; self.W=0.; self.children={}; self.state=None; self.reward=0.
    def Q(self): return self.W/self.N if self.N>0 else 0.
def run_mcts(model, pos, sims=80, c1=1.25, c2=19652):
    with torch.no_grad():
        s0 = model.repr(oh(pos, N).unsqueeze(0)); pl, v0 = model.pred(s0); pri = F.softmax(pl, -1).squeeze(0)
    root = Node(0.); root.state = s0
    for a in range(A): root.children[a] = Node(float(pri[a]))
    for _ in range(sims):
        node, path = root, [root]
        while node.children and all(c.state is not None for c in node.children.values()):
            tot = sum(c.N for c in node.children.values()); best, ba = -1e9, 0
            for a, c in node.children.items():
                pb = (math.log((tot+c2+1)/c2)+c1)*c.prior*math.sqrt(tot)/(1+c.N)   # pUCT (Eq. 2)
                if c.Q()+pb > best: best, ba = c.Q()+pb, a
            node = node.children[ba]; path.append(node)
        tot = sum(c.N for c in node.children.values()) if node.children else 0; best, ba = -1e9, 0
        for a, c in node.children.items():
            pb = (math.log((tot+c2+1)/c2)+c1)*c.prior*math.sqrt(tot+1)/(1+c.N)
            if c.Q()+pb > best: best, ba = c.Q()+pb, a
        leaf = node.children[ba]
        with torch.no_grad():
            ns, r = model.dyn(node.state, oh(ba, A).unsqueeze(0)); pl, lv = model.pred(ns); p = F.softmax(pl, -1).squeeze(0)
        leaf.state, leaf.reward = ns, float(r)
        for a in range(A): leaf.children[a] = Node(float(p[a]))
        G = float(lv)
        for nd in reversed(path + [leaf]):                                    # back up discounted value
            nd.N += 1; nd.W += G; G = nd.reward + GAMMA*G
    return [root.children[a].N for a in range(A)], float(v0)

# ===== WORKED EXAMPLE: one latent rollout step from start (must match the lesson) =====
def r3(t): return np.round(t.detach().numpy().ravel(), 3)
with torch.no_grad():
    s0 = mz.repr(oh(START, N).unsqueeze(0)); pl0, v0 = mz.pred(s0)
    s1R, r1R = mz.dyn(s0, oh(1, A).unsqueeze(0)); plR, vR = mz.pred(s1R)
    s1L, r1L = mz.dyn(s0, oh(0, A).unsqueeze(0)); plL, vL = mz.pred(s1L)
print("s0 = h(obs) =", r3(s0))
print("f(s0): p0 =", r3(F.softmax(pl0,-1)), " v0 =", round(v0.item(),3))     # ~[0.49,0.51], 0.255 -> prior is a coin flip
print("g(s0,RIGHT): r^1 =", round(r1R.item(),3), " then f-> v^1 =", round(vR.item(),3))  # value rises (~0.445)
print("g(s0,LEFT) : r^1 =", round(r1L.item(),3), " then f-> v^1 =", round(vL.item(),3))  # value falls (~0.162)
vis, _ = run_mcts(mz, START, 80)
print("MCTS visits [LEFT,RIGHT] =", vis, "-> choose", "RIGHT" if vis[1] > vis[0] else "LEFT")  # ~[5,75] RIGHT

# ===== ABLATION: train with unroll depth 1 (g never rolled forward) -> planning collapses =====
def evaluate(model, sims=80, trials=20):
    succ = 0
    for t in range(trials):
        np.random.seed(2000+t); p = START
        for step in range(30):
            v, _ = run_mcts(model, p, sims); a = int(np.argmax(v)); p, r, d = true_step(p, a)
            if d: succ += (p == GOAL); break
    return succ/trials
abl = train(unroll=False)
print("FULL MuZero  (K=4) success:", evaluate(mz))    # 1.0
print("ABLATION (no unroll) success:", evaluate(abl)) # 0.0 -- latent model can't roll forward, planning fails

## Visualize the data & results

_Does planning inside a LEARNED latent model (full MuZero, dynamics unrolled K=4) solve the corridor, and does the ablation that never unrolls the dynamics (K=1) collapse? Self-play is unbiased 50/50, so the policy prior gives no direction — only planning can win. We train both with identical nets/optimizer/seed and measure goal-reaching success as training proceeds._

In [ ]:
# How the two curves are produced (full loop in the CODE cell):
#   full = train(unroll=True,  iters=4000)   # green: g unrolled K=4 -> learns latent transitions
#   abl  = train(unroll=False, iters=4000)   # red:   g never rolled forward -> latent model is meaningless
# Every ~800 iters we run greedy MCTS rollouts from the start and record success.
#   evaluate(full) -> 1.0   (reaches goal in optimal 3 steps)
#   evaluate(abl)  -> 0.0   (planning collapses; 50/50 prior gives no fallback signal)
# Numbers are from our own toy run; the paper reports Atari/Go/chess/shogi results, not these.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** State, with signatures, the three functions $h$, $g$, $f$ and say which one MCTS calls to expand a new node.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write $s^0=h(o_1,\dots,o_t)$. — _$h$ encodes observations into the root hidden state._
- Write $r^k,s^k=g(s^{k-1},a^k)$. — _$g$ is the learned dynamics: reward + next latent state._
- Write $p^k,v^k=f(s^k)$. — _$f$ reads policy + value off a hidden state._

**Answer:** $h:\text{obs}\to s^0$, $g:(s,a)\to(r,s')$, $f:s\to(p,v)$. To expand a leaf, MCTS calls BOTH $g$ (for the child's reward and hidden state) and $f$ (for the child's prior policy and value). Selection uses pUCT with $Q$, $P$, $N$.

</details>

**Problem 2.** Why can MuZero plan well even though its hidden states need not match the true environment state?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List what the search actually reads from the model. — _Only $g$'s reward/next-state and $f$'s policy/value — never the raw state._
- Argue value-equivalence. — _If reward and value predictions match the real ones for every action sequence, the search tree is the same._

**Answer:** Because MCTS only consumes the model's predicted rewards, values, and policies. If those match what the true environment would yield, the constructed tree — and thus the chosen move — is identical, regardless of what the latent states "mean". The model is trained to be value-equivalent, not state-accurate.

</details>

**Problem 3.** ABLATION: train the toy model with unroll depth $K=1$ (encode the root, train $f$ there, but NEVER unroll $g$ forward during training). Predict and explain what happens to planning, and contrast with full MuZero.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the self-play policy is 50/50, so the prior $p^0$ is ~uninformative. — _Directional signal cannot come from the policy head._
- With $K=1$, the dynamics $g$ is never trained to produce sensible next states/rewards. — _Its only gradient came from depth-0; rolled one step out, $g$'s output is meaningless._
- MCTS expands with this broken $g$. — _The values backed up from imagined leaves are noise, so the search cannot tell RIGHT from LEFT._
- Compare to full MuZero ($K=4$). — _There $g$ learns the latent transition, so imagined value rises toward the goal and the search finds RIGHT._

**Answer:** Planning collapses. In our run, FULL MuZero (unroll $K=4$) reaches the goal on 100% of trials in the optimal 3 steps, while the $K=1$ ablation succeeds 0% of the time — its latent model cannot roll forward, so MCTS plans through garbage and, with no directional prior to fall back on, never reaches the goal. This isolates the paper's core claim: the value comes from planning inside a learned dynamics model, not from the policy prior. (Numbers are our small toy run, not the paper's.)

</details>